In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import tqdm
import sys
import os

In [ ]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [ ]:
sys.path.append(".")

In [ ]:
from utils.datasets import SimpleSet, BerlinSparqlBenchmark
from utils.dbs.qlever import QleverDB
from utils.dbs.fuseki import FusekiDB
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import pandas as pd
import numpy as np
from utils.datasets.base_dataset import DataTensor

2026-04-28 16:13:24,262 - INFO - Loading faiss with AVX512 support.
2026-04-28 16:13:24,263 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-04-28 16:13:24,264 - INFO - Loading faiss with AVX2 support.
2026-04-28 16:13:24,265 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-04-28 16:13:24,265 - INFO - Loading faiss.
2026-04-28 16:13:24,293 - INFO - Successfully loaded faiss.


In [ ]:
powers = np.arange(0, 6)  # extend on a more powerful machine
sizes = 10**powers

In [ ]:
datasets: dict[int, BerlinSparqlBenchmark] = {}
raw_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Running BDSDM generation for size {size}...")
    dataset = BerlinSparqlBenchmark(base_dir=Path(f"./data/bsbm_{power}"), n=size)
    dataset.setup()
    datasets[power] = dataset
    # raw_sizes[power] = dataset.get_triple_count()

2026-04-28 16:13:24,423 - INFO - BSBM dataset already exists in data/bsbm_0, skipping generation
2026-04-28 16:13:24,424 - INFO - BSBM dataset already exists in data/bsbm_1, skipping generation
2026-04-28 16:13:24,425 - INFO - BSBM dataset already exists in data/bsbm_2, skipping generation
2026-04-28 16:13:24,425 - INFO - BSBM dataset already exists in data/bsbm_3, skipping generation
2026-04-28 16:13:24,426 - INFO - BSBM dataset already exists in data/bsbm_4, skipping generation
2026-04-28 16:13:24,426 - INFO - BSBM dataset already exists in data/bsbm_5, skipping generation


Running BDSDM generation for size 1...
Running BDSDM generation for size 10...
Running BDSDM generation for size 100...
Running BDSDM generation for size 1000...
Running BDSDM generation for size 10000...
Running BDSDM generation for size 100000...


In [ ]:
encoded_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Encoding dataset of size {size}/power {power}...")
    dataset = datasets[power]
    encoded_sizes[power] = dataset.encode(encoding_model)
    #  dataset.get_triple_count(encoded=True)

Encoding dataset of size 1/power 0...
Encoded TTL file already exists at data/bsbm_0/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10/power 1...
Encoded TTL file already exists at data/bsbm_1/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100/power 2...
Encoded TTL file already exists at data/bsbm_2/dataset_encoded.nt, skipping encoding
Encoding dataset of size 1000/power 3...
Encoded TTL file already exists at data/bsbm_3/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10000/power 4...
Encoded TTL file already exists at data/bsbm_4/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100000/power 5...
Encoded TTL file already exists at data/bsbm_5/dataset_encoded.nt, skipping encoding


## BSBM queries

### Simple Use case: 
find 10 products with a specific encoded label


### Complex Use case: 
For a specific product find 10 other similar products via their product label. 


In [ ]:
test_label = "house furniture storage container"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
test_tensor.to_literal().n3()

'"{\\"data\\": [0.001923849806189537, 0.059069667011499405, -0.06259278953075409, -0.008788925595581532, 0.08017760515213013, 0.0004897266626358032, 0.049011558294296265, -0.016156358644366264, -0.04537816718220711, 0.011929630301892757, -0.003551364177837968, -0.0034783161245286465, 0.023264417424798012, 0.02449001744389534, -0.023595571517944336, -0.073823943734169, 0.014226873405277729, -0.00022305997845251113, -0.03111756220459938, 0.07163400948047638, -0.04559392109513283, 0.044428009539842606, 0.0004388350062072277, -0.004044292028993368, 0.05173112079501152, 0.08436278998851776, -0.028667306527495384, -0.032494205981492996, 0.058949727565050125, -0.0051073553040623665, 0.09506019204854965, -0.029170090332627296, -0.07172352075576782, 0.036426812410354614, 0.01788988709449768, 0.07774496078491211, -0.01058284193277359, -0.02719942107796669, -0.014839782379567623, 0.004996407311409712, -0.05195301026105881, -0.01563340425491333, -0.0004883587826043367, 0.012572054751217365, -0.043

In [ ]:
base_bsbm_set = datasets[4]
db = QleverDBNative(
    id="test",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
)
possible_queries = db.get_queries(test_tensor)
print(possible_queries)


2026-04-28 16:13:25,128 - WARNING - Killing any existing process using port 26043 before starting the server
2026-04-28 16:13:25,149 - ERROR - Command failed with return code 1
2026-04-28 16:13:25,150 - INFO - Initialized QLeverDBNative with id=test, port_id=26043, dataset=bsbm, name=QLever Native (Extended), use_encoded_ttl=True, endpoint=http://localhost:26043/test-with-tidx/sparql


{<QUERY_DIFFICULTY.EASY: 'easy'>: {<QUERY_TYPE.EMBEDDED: 'embedded'>: '\nPREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>\nPREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>\nPREFIX dtf: <https://w3id.org/rdf-tensor/functions#>\nPREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>\nPREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>\nSELECT DISTINCT ?product  ?vector ?dist \nWHERE {\n?product rdf:label_embedding ?vector .\n?product bsbmv:productFeature ?feat .\nBIND(dtf:dotProduct(?vector, "{\\"data\\": [0.001923849806189537, 0.059069667011499405, -0.06259278953075409, -0.008788925595581532, 0.08017760515213013, 0.0004897266626358032, 0.049011558294296265, -0.016156358644366264, -0.04537816718220711, 0.011929630301892757, -0.003551364177837968, -0.0034783161245286465, 0.023264417424798012, 0.02449001744389534, -0.023595571517944336, -0.073823943734169, 0.014226873405277729, -0.00022305997845251113, -0.03111756220459938, 0.0716340094804763

In [ ]:
from typing import Literal


def get_query_index_variations(
    embedding: DataTensor,
    numNN: int = 1,
    searchK: int = 16,
    mode: Literal["naive", "hsnw", "ivf"] = "naive",
) -> str:
    args = f"""
    _:config tensorIndex:numNN {numNN} ;
    tensorIndex:left ?query_vector ;
    tensorIndex:bindDistance ?dist ;
    tensorIndex:payload ?product ;
    tensorIndex:algorithm tensorIndex:{mode} ;
    tensorIndex:distance tensorIndex:dot ;
    """
    if mode != "naive":
        args += f"""
        tensorIndex:experimentalRightCacheName "easy_index_{mode}_{searchK}" ;
        tensorIndex:searchK {searchK} ;
        """
    if mode == "ivf":
        args += """
        tensorIndex:kIVF 16 ;
        """
    return f"""
PREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>
PREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT DISTINCT ?product  ?vector ?dist
WHERE {{
SERVICE tensorIndex: {{
        {args}
    tensorIndex:right ?vector .
       {{
            {{
                SELECT DISTINCT ?product ?vector WHERE {{
                    ?product rdf:label_embedding ?vector .
                    ?product bsbmv:productFeature ?feat .
                }} GROUP BY ?product ?vector
            }}
        }}
    }}
    VALUES (?query_vector) {{ ({embedding.to_literal().n3()}) }}
}} ORDER BY DESC(?dist)
LIMIT 16
"""


In [ ]:
# 1. get reference results using naive
k = 16
with db:
    query = get_query_index_variations(test_tensor, numNN=k, mode="naive")
    print("Running naive query...")
    results_naive = db.query(query)
results_naive

2026-04-28 16:13:25,265 - INFO - Logging QLever setup to scratch/bsbm/bsbm_4/db/test-with-tidx/test-with-tidx_run.log
2026-04-28 16:13:25,266 - INFO - Loading dataset into QLever server from data/bsbm_4/dataset.nt
2026-04-28 16:13:25,267 - WARNING - DB directory scratch/bsbm/bsbm_4/db/test-with-tidx already exists!
2026-04-28 16:13:25,267 - INFO - Stopping server!
2026-04-28 16:13:25,288 - ERROR - Command failed with return code 1
2026-04-28 16:13:25,288 - INFO - Starting QLever server on port 26043
2026-04-28 16:13:25,288 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-28 16:13:25,298 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-04-28 16:13:26,310 - INFO - Server is up and responding to queries
2026-04-28 16:13:26,382 - INFO - Stopping server!
2026-04-28 16:13:26,460 - ERROR - Command failed with return c

Running naive query...


,product,vector,dist
0,bsbmi:dataFromProducer42/Product1985,"{""data"": [-0.053947802633047104, 0.09995079785...",0.4656711816788
1,bsbmi:dataFromProducer204/Product9871,"{""data"": [-0.019114747643470764, 0.04421398043...",0.4265947937965
2,bsbmi:dataFromProducer126/Product6035,"{""data"": [-0.027103513479232788, 0.09437111020...",0.4230341911316
3,bsbmi:dataFromProducer90/Product4341,"{""data"": [-0.046234358102083206, 0.02935838326...",0.4177694320679
4,bsbmi:dataFromProducer107/Product5062,"{""data"": [-0.03348818048834801, 0.051626261323...",0.4087207317352
5,bsbmi:dataFromProducer58/Product2673,"{""data"": [-0.01993369869887829, 0.017058538272...",0.4059612154961
6,bsbmi:dataFromProducer103/Product4854,"{""data"": [0.03547070920467377, 0.0046439259313...",0.3998067677021
7,bsbmi:dataFromProducer77/Product3653,"{""data"": [-0.05063876882195473, 0.055860601365...",0.3955109715462
8,bsbmi:dataFromProducer189/Product9168,"{""data"": [-0.02634534053504467, 0.026425572112...",0.391575217247
9,bsbmi:dataFromProducer37/Product1658,"{""data"": [-0.10466377437114716, 0.054606124758...",0.3833258748055


In [ ]:
# 2. get results using ivf
with db:
    query = get_query_index_variations(test_tensor, numNN=k, mode="ivf", searchK=k)
    print("Running ivf query...")
    results_idx = db.query(query)
results_idx

2026-04-28 16:13:26,517 - INFO - Logging QLever setup to scratch/bsbm/bsbm_4/db/test-with-tidx/test-with-tidx_run.log
2026-04-28 16:13:26,517 - INFO - Loading dataset into QLever server from data/bsbm_4/dataset.nt
2026-04-28 16:13:26,518 - WARNING - DB directory scratch/bsbm/bsbm_4/db/test-with-tidx already exists!
2026-04-28 16:13:26,518 - INFO - Stopping server!
2026-04-28 16:13:26,593 - ERROR - Command failed with return code 1
2026-04-28 16:13:26,593 - INFO - Starting QLever server on port 26043
2026-04-28 16:13:26,594 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-28 16:13:26,595 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-04-28 16:13:27,606 - INFO - Server is up and responding to queries
2026-04-28 16:13:27,787 - INFO - Stopping server!


Running ivf query...


2026-04-28 16:13:27,870 - ERROR - Command failed with return code 1


,product,vector,dist
0,bsbmi:dataFromProducer42/Product1985,"{""data"": [-0.053947802633047104, 0.09995079785...",0.4656711220741
1,bsbmi:dataFromProducer204/Product9871,"{""data"": [-0.019114747643470764, 0.04421398043...",0.4265947937965
2,bsbmi:dataFromProducer126/Product6035,"{""data"": [-0.027103513479232788, 0.09437111020...",0.4230341911316
3,bsbmi:dataFromProducer90/Product4341,"{""data"": [-0.046234358102083206, 0.02935838326...",0.4177694320679
4,bsbmi:dataFromProducer107/Product5062,"{""data"": [-0.03348818048834801, 0.051626261323...",0.4087207317352
5,bsbmi:dataFromProducer58/Product2673,"{""data"": [-0.01993369869887829, 0.017058538272...",0.4059612154961
6,bsbmi:dataFromProducer103/Product4854,"{""data"": [0.03547070920467377, 0.0046439259313...",0.3998067378998
7,bsbmi:dataFromProducer77/Product3653,"{""data"": [-0.05063876882195473, 0.055860601365...",0.3955109715462
8,bsbmi:dataFromProducer189/Product9168,"{""data"": [-0.02634534053504467, 0.026425572112...",0.3915751874447
9,bsbmi:dataFromProducer37/Product1658,"{""data"": [-0.10466377437114716, 0.054606124758...",0.3833258748055


In [ ]:
import time

from utils.helpers import recall_at_k, precision_at_k, ndcgscore_query

In [ ]:
recall_at_k(results_idx, results_naive, k=16)

1.0

In [ ]:
out_dir = Path("./scratch") / "results" / "bsbm" / "index_tests"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:


with db:
    runs = 256
    search_ks = {
        "hnsw": np.arange(0, 33, 4).tolist(),
        "ivf": np.arange(0, 17, 2).tolist(),
    }
    for m, search_k_list in search_ks.items():
        search_ks[m] = [1] + search_k_list[1:]

    modes = ["hnsw", "ivf"]
    total = runs * len(modes) * sum(len(s) for s in search_ks.values())
    g = tqdm.tqdm(total=total, desc="Running index queries")

    def run_index_query(
        test_tensor: DataTensor, mode: Literal["naive", "hnsw", "ivf"], search_k: int
    ):
        query = get_query_index_variations(
            test_tensor, numNN=k, searchK=search_k, mode=mode
        )
        return db.raw_query(query)

    timing_results = []
    for mode in modes:
        for search_k in search_ks[mode]:
            run_index_query(test_tensor, mode, search_k)  # warmup
            for run in range(runs):
                noised_tensor = DataTensor.from_numpy(
                    test_tensor.data + np.random.normal(scale=1, size=test_tensor.shape)
                )
                reference_results = db.q_to_df_values(
                    run_index_query(noised_tensor, "naive", search_k)
                )
                g.update(1)
                g.set_description(
                    f"Running {mode} query, run {run + 1}/{runs}, searchK={search_k}"
                )
                result = {}
                start = time.time()
                resp = run_index_query(noised_tensor, mode, search_k)
                end = time.time()
                results_df = db.q_to_df_values(resp)

                result["mode"] = mode
                result["run"] = run
                result["time"] = end - start
                result["searchK"] = search_k
                result["ndcg_score"] = ndcgscore_query(
                    results_df, reference_results, k=k
                )
                result["recall_score"] = recall_at_k(results_df, reference_results, k=k)
                timing_results.append(result)
    g.close()

2026-04-28 16:13:28,085 - INFO - Logging QLever setup to scratch/bsbm/bsbm_4/db/test-with-tidx/test-with-tidx_run.log
2026-04-28 16:13:28,086 - INFO - Loading dataset into QLever server from data/bsbm_4/dataset.nt
2026-04-28 16:13:28,086 - WARNING - DB directory scratch/bsbm/bsbm_4/db/test-with-tidx already exists!
2026-04-28 16:13:28,087 - INFO - Stopping server!
2026-04-28 16:13:28,164 - ERROR - Command failed with return code 1
2026-04-28 16:13:28,164 - INFO - Starting QLever server on port 26043
2026-04-28 16:13:28,165 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-04-28 16:13:28,166 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-04-28 16:13:29,178 - INFO - Server is up and responding to queries
Running ivf query, run 256/256, searchK=16:  50%|█████     | 4608/9216 [08:10<08:10,  9.40it/s] 
2026-04-28 16:2

In [ ]:
timing_results_df = pd.DataFrame(timing_results)
timing_results_df.to_csv(out_dir / "index_query_timings.csv", index=False)


In [ ]:
noised_tensor

DataTensor(data=[-0.29541585659025676, 1.7479325210583716, -0.1251014632094179, 1.235369663954073, 0.43863498019171754, 1.9786149169237732, 0.9071609802115765, 1.8952518483350937, 0.8151660685294816, 0.01962228867984513, 1.406735179291108, -0.8958258495698949, -1.4180430310850831, -0.8247317383146132, 0.7575070498494731, 0.05743172636420091, -0.8236178967541068, -1.1218212258093179, -0.6821265882603661, -1.5119489986142811, -0.31409410389051684, 0.02162648407712839, -0.050259223087955295, 0.47801894013951346, 1.1367245420213057, 0.20770097127577786, -2.664950895393883, -0.6811459519058084, 0.31745908784929333, -0.42571504703414903, -0.29694626619931347, 1.66616699524499, -0.7435331743099688, 0.8758314543783867, -0.9234127845397713, -1.4620426635880057, -0.7217010687832609, 1.0447647067142734, -2.3030090406137425, 0.34879685741390676, -0.07691368846828889, -0.6281559162054275, 1.7467379073739289, -0.08185912897520504, 2.1270793219101742, 1.308618033577409, -0.6084968351851459, -1.041049